# 02 · Robust and clustered inference

**Module:** 4 · **Time:** 50 minutes · **Prerequisites:** 01 OLS in statsmodels

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/modules/04_econometrics/02_standard_errors_and_tests.ipynb) · [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ningzhanglab/Python-for-Beginners/main?filepath=modules%2F04_econometrics%2F02_standard_errors_and_tests.ipynb)

**In Colab choose File → Save a copy in Drive before editing.** Badges target `main` after merge; this is a private repository and draft PR.

**Completed solution edition.**

## Learning goals

- **D**istinguish conventional, HC3, and cluster-robust standard errors.
- **F**it a clustered OLS covariance estimator with explicit groups.
- **E**xplain why clustering and degrees of freedom matter for inference.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

rng = np.random.default_rng(512)
n_obs = 150
cluster_id = np.repeat(np.arange(30), 5)
schooling = rng.integers(10, 19, size=n_obs)
experience = rng.integers(0, 25, size=n_obs)
cluster_shocks = rng.normal(0, 2, size=30)
person_shocks = rng.normal(0, 3, size=n_obs)

In [ ]:
wage_index = (5 + 1.6 * schooling + 0.5 * experience
              + cluster_shocks[cluster_id] + person_shocks)
survey = pd.DataFrame({"wage_index": wage_index,
                       "schooling": schooling, "experience": experience,
                       "cluster_id": cluster_id})
print("Fictional workers:", len(survey))
print(survey.head(3))

## Why this matters

Point estimates alone do not describe uncertainty. Heteroskedasticity can invalidate conventional OLS standard-error formulas; repeated observations within a group may also have correlated disturbances. Our fictional dataset includes a shared group shock, so group-level clustering is a useful demonstration. No estimator alone makes the coefficients causal.

## Conventional versus HC3 standard errors

HC3 is a heteroskedasticity-consistent covariance estimator with a leverage adjustment. It need not always be numerically larger than the default model-based standard error.

In [ ]:
formula = "wage_index ~ schooling + experience"
classic = smf.ols(formula, data=survey).fit()
hc3 = smf.ols(formula, data=survey).fit(cov_type="HC3")
comparison = pd.DataFrame({"conventional": classic.bse, "HC3": hc3.bse})
print(comparison.round(3))

## Cluster by the data-generating group

The dataset has 30 groups of five fictional workers with shared group shocks. Pass the cluster labels for **exactly the estimation sample**, avoiding misalignment after dropping missing observations.

In [ ]:
clustered = smf.ols(formula, data=survey).fit(
    cov_type="cluster", cov_kwds={"groups": survey["cluster_id"]},
    use_t=True
)
print("Clusters:", survey["cluster_id"].nunique())
print("Clustered standard errors:")
print(clustered.bse.round(3))

## Compare confidence intervals

Coefficients from the same linear model are unchanged by a covariance choice; uncertainty intervals can change. With `use_t=True`, the cluster fit uses a finite-sample t reference; few clusters may still make conventional clustered inference unreliable.

In [ ]:
se_table = pd.DataFrame({"OLS": classic.bse,
                         "HC3": hc3.bse, "Cluster": clustered.bse})
print(se_table.round(3))
print("Clustered schooling interval:")
print(clustered.conf_int().loc["schooling"])

## Test a coefficient against zero

Read the test statistic and p-value from the result object, while keeping the null hypothesis, model and covariance estimator in view. Do not interpret p-values as probabilities that a hypothesis is true.

In [ ]:
test_result = clustered.t_test("schooling = 0")
print("Cluster-based t test:", test_result.tvalue)
print("Reported p-value:", test_result.pvalue)

## If you know Stata/R

| Inference | Python | Stata | R |
| --- | --- | --- | --- |
| Conventional OLS | `smf.ols(...).fit()` | `reg y x` | `lm(y ~ x)` |
| HC3 covariance | `.fit(cov_type='HC3')` | `vce(hc3)` (verify Stata version) | `sandwich::vcovHC(m,type='HC3')` |
| Group clustered | `.fit(cov_type='cluster',cov_kwds={'groups':g})` | `vce(cluster group)` | `sandwich::vcovCL(m,cluster=~group)` |

Clustered uncertainty depends on the number of independent groups. The reported interval and reference distribution are choices that must be documented.

## Try it

Exercises begin with `None` in the learner notebook. Checks offer hints rather than raising errors, while completed solutions must pass every check.

### ★ Exercise 1 — Count independent clusters

Assign `cluster_count` to the number of unique group labels in the survey.

In [ ]:
cluster_count = int(survey["cluster_id"].nunique())

In [ ]:
if isinstance(cluster_count, (int, np.integer)) and cluster_count == 30:
    print("✓ Number of clusters correct")
else:
    print("Hint: use nunique() on the cluster_id column.")

### ★★ Exercise 2 — Fit an HC3 model

Create `hc3_fit` with heteroskedasticity-consistent HC3 inference for `formula`.

In [ ]:
hc3_fit = smf.ols(formula, data=survey).fit(cov_type="HC3")

In [ ]:
if (getattr(hc3_fit, "cov_type", None) == "HC3" and
    np.allclose(hc3_fit.params.to_numpy(), classic.params.to_numpy())):
    print("✓ HC3 covariance applied")
else:
    print("Hint: pass cov_type='HC3' when fitting the OLS model.")

### ★★★ Exercise 3 — Fit a clustered model

Create `cluster_fit`, clustering on the `cluster_id` column and using a t reference distribution.

In [ ]:
cluster_fit = smf.ols(formula, data=survey).fit(
    cov_type="cluster", cov_kwds={"groups": survey["cluster_id"]},
    use_t=True
)

In [ ]:
if (getattr(cluster_fit, "cov_type", None) == "cluster" and
    np.isfinite(cluster_fit.bse.to_numpy()).all() and cluster_fit.use_t):
    print("✓ Cluster-robust inference applied")
else:
    print("Hint: use cov_type='cluster', cov_kwds={'groups': ...}, use_t=True.")

### ★★★ Exercise 4 — Extract a clustered 95% interval

Create `schooling_interval` as the two-element pandas Series from the clustered model for schooling.

In [ ]:
schooling_interval = cluster_fit.conf_int().loc["schooling"]

In [ ]:
expected_interval = clustered.conf_int().loc["schooling"]
if (isinstance(schooling_interval, pd.Series) and
    np.allclose(schooling_interval.to_numpy(), expected_interval.to_numpy())):
    print("✓ Clustered interval extracted")
else:
    print("Hint: use .conf_int().loc['schooling'] on your clustered result.")

## Common errors

- Cluster labels must align one-for-one with the model's estimation sample after missing-value handling.
- Cluster-robust standard errors are not automatically larger than conventional standard errors.
- A small number of clusters can make normal or t approximations unreliable.
- Robust inference does not address omitted-variable bias or nonrandom treatment assignment.

## Recap

- Choosing a covariance estimator changes uncertainty, not the OLS point estimate.
- HC3 addresses heteroskedasticity; clustering permits within-group error dependence.
- Always report the clustering level and number of clusters.

## Next steps

Continue to [Regression tables](03_regression_tables.ipynb). See the [official statsmodels robust covariance reference](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.RegressionResults.get_robustcov_results.html).

[Report a content error](https://github.com/ningzhanglab/Python-for-Beginners/issues/new?template=content_error.md) with this lesson path.